In [ ]:
#pip install geopandas

In [5]:

pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [6]:
import geopandas as gpd
import pandas as pd

In [7]:
import os
os.path.exists(r"C:/data/GitHub/NSCCProject-TeamCharlie/scripts/python/Yashaswi/lcd_000a21a_e/")


True

In [16]:
stations = pd.read_excel("../../../data/clean/Yashaswi/station.xlsx", sheet_name="Station_All")

In [17]:
print(stations.columns.tolist())

['Station ID', 'Latitude', 'Longitude']


In [27]:
import geopandas as gpd
import pandas as pd

# Load Census Divisions
cd = gpd.read_file(
    r"C:/data/GitHub/NSCCProject-TeamCharlie/scripts/python/Yashaswi/lcd_000a21a_e/lcd_000a21a_e.shp"
).to_crs(epsg=4326)

# Load station data (explicit sheet)
stations = pd.read_excel(
    "../../../data/clean/Yashaswi/station.xlsx",
    sheet_name="Station_All"
)

print(stations.columns.tolist())  # ✅ sanity check

# Convert stations to GeoDataFrame (FIXED)
stations_gdf = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(
        stations["Longitude"],
        stations["Latitude"]
    ),
    crs="EPSG:4326"
)

# Spatial join
joined = gpd.sjoin(
    stations_gdf,
    cd,
    how="left",
    predicate="within"
)


['Station ID', 'Latitude', 'Longitude']


In [28]:
province_lookup = {
    "10": "Newfoundland and Labrador",
    "11": "Prince Edward Island",
    "12": "Nova Scotia",
    "13": "New Brunswick",
    "24": "Quebec",
    "35": "Ontario",
    "46": "Manitoba",
    "47": "Saskatchewan",
    "48": "Alberta",
    "59": "British Columbia",
    "60": "Yukon",
    "61": "Northwest Territories",
    "62": "Nunavut"
}

In [29]:
joined["PRUID"] = joined["PRUID"].astype(str)
joined["PRNAME"] = joined["PRUID"].map(province_lookup)

In [30]:
cols = [
    "Station ID",
    "Latitude",
    "Longitude",
    "CDUID",
    "CDNAME",
    "PRUID",
    "PRNAME"
]

joined[cols].to_csv(
    "stations_with_census_divisions.csv",
    index=False
)

In [20]:
print(joined.columns.tolist())

['Station ID', 'Latitude', 'Longitude', 'geometry', 'index_right', 'CDUID', 'DGUID', 'CDNAME', 'CDTYPE', 'LANDAREA', 'PRUID']


In [32]:
import geopandas as gpd
import pandas as pd

# =====================================================
# 1. LOAD CMA / CA BOUNDARIES (CITY / REGIONAL NAMES)
# =====================================================
cma = gpd.read_file(
    r"C:/data/GitHub/NSCCProject-TeamCharlie/scripts/python/Yashaswi/lcma000a21a_e/lcma000a21a_e.shp"
).to_crs(epsg=4326)

# Optional sanity check
print(cma.columns.tolist())
# Expect columns like: CMAUID, CMANAME, PRUID, DGUID, etc.

# =====================================================
# 2. LOAD STATION DATA (EXCEL)
# =====================================================
stations = pd.read_excel(
    "../../../data/clean/Yashaswi/station.xlsx",
    sheet_name="Station_All"
)

print(stations.columns.tolist())
# Expected: ['Station ID', 'Latitude', 'Longitude']

# =====================================================
# 3. CONVERT STATIONS TO GEODATAFRAME
# =====================================================
stations_gdf = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(
        stations["Longitude"],
        stations["Latitude"]
    ),
    crs="EPSG:4326"
)

# =====================================================
# 4. SPATIAL JOIN (STATION → CMA / CA)
# =====================================================
joined = gpd.sjoin(
    stations_gdf,
    cma,
    how="left",
    predicate="within"
)

print(joined.columns.tolist())

# =====================================================
# 5. EXPORT CLEAN OUTPUT
# =====================================================
cols = [
    "Station ID",
    "Latitude",
    "Longitude",
    "CMAUID",
    "CMANAME",
    "PRUID"
]

joined[cols].to_csv(
    "stations_with_CMA_CA.csv",
    index=False
)

['CMAUID', 'CMAPUID', 'DGUID', 'DGUIDP', 'CMANAME', 'CMATYPE', 'LANDAREA', 'PRUID', 'geometry']
['Station ID', 'Latitude', 'Longitude']
['Station ID', 'Latitude', 'Longitude', 'geometry', 'index_right', 'CMAUID', 'CMAPUID', 'DGUID', 'DGUIDP', 'CMANAME', 'CMATYPE', 'LANDAREA', 'PRUID']


In [33]:
province_lookup = {
    "10": "Newfoundland and Labrador",
    "11": "Prince Edward Island",
    "12": "Nova Scotia",
    "13": "New Brunswick",
    "24": "Quebec",
    "35": "Ontario",
    "46": "Manitoba",
    "47": "Saskatchewan",
    "48": "Alberta",
    "59": "British Columbia",
    "60": "Yukon",
    "61": "Northwest Territories",
    "62": "Nunavut"
}

In [35]:
import geopandas as gpd
import pandas as pd

# =====================================================
# 1. LOAD CENSUS SUBDIVISIONS (MUNICIPAL / REGIONAL LEVEL)
# =====================================================
csd = gpd.read_file(
    r"C:/data/GitHub/NSCCProject-TeamCharlie/scripts/python/Yashaswi/lcsd000a21a_e/lcsd000a21a_e.shp"
).to_crs(epsg=4326)

print(csd.columns.tolist())
# Expect: CSDUID, CSDNAME, PRUID, CDUID, geometry

# =====================================================
# 2. LOAD STATION DATA
# =====================================================
stations = pd.read_excel(
    "../../../data/clean/Yashaswi/station.xlsx",
    sheet_name="Station_All"
)

stations_gdf = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(
        stations["Longitude"],
        stations["Latitude"]
    ),
    crs="EPSG:4326"
)

# =====================================================
# 3. SPATIAL JOIN (STATION → MUNICIPAL / REGIONAL DISTRICT)
# =====================================================
joined = gpd.sjoin(
    stations_gdf,
    csd[["CSDUID", "CSDNAME", "PRUID", "geometry"]],
    how="left",
    predicate="within"
)

# =====================================================
# 4. EXPORT FINAL DATASET
# =====================================================
final_cols = [
    "Station ID",
    "Latitude",
    "Longitude",
    "CSDUID",
    "CSDNAME",
    "PRUID"
]

joined[final_cols].to_csv(
    "stations_with_municipal_regions.csv",
    index=False
)

['CSDUID', 'DGUID', 'CSDNAME', 'CSDTYPE', 'LANDAREA', 'PRUID', 'geometry']


In [36]:
joined["PRUID"] = joined["PRUID"].astype(str)
joined["Province"] = joined["PRUID"].map(province_lookup)


In [37]:
final_cols = [
    "Station ID",
    "Latitude",
    "Longitude",
    "CSDUID",
    "CSDNAME",     # e.g. Halifax Regional Municipality
    "Province"     # e.g. Nova Scotia
]

joined[final_cols].to_csv(
    "stations_with_municipality_and_province.csv",
    index=False
)
